In [1]:
import load_data as jtd
import pandas as pd
import numpy as np
import ast

import candidates as ds
import priority_queue as pq
import subgroups as s
import refinements as rf

In [2]:
df_library = pd.read_csv('algebrakit_firstbatch.csv')
collection_attributes = ['skillName', 'skillsTodo', 'stepType']

for colatt in collection_attributes:
    df_library[colatt] = df_library[colatt].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

df_library.head()

,exerciseID,sessionID,sessionSuccess,creationTimestamp,finished,marksEarned,marksTotal,skillName,stepType,errors,...,result.scoring.usedAttempts,result.scoring.failed,eventID,eventType,eventTags,eventAnnotations,exerciseStatusAfterEvent,progressAfterEvent,inputStatus,skillsTodo
0,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[nan],...,0.0,False,1.740471e+12,EVALUATE,[],"[{'type': 'INPUT_EXPRESSION', 'expr': '$n=4\\c...",FINISHED,1.0,FINISHED,NaN
1,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[nan],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[nan],...,2.0,True,1.740472e+12,EVALUATE,[],"[{'type': 'INPUT_EXPRESSION', 'expr': '$u=t^{3...",CORRECT,0.0,CORRECT,NaN
3,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[nan],...,2.0,True,1.740569e+12,EVALUATE,[],"[{'type': 'INPUT_EXPRESSION', 'expr': '$t^{2}\...",ERROR,0.0,ERROR,NaN
4,9f4a675e-c2b3-4e11-91cc-4445951bab60,027d7123-7fd5-423f-8c45-94206bca6a45,True,1740470686,False,1.0,4.0,[FractionArithmeticMultiplicationProper],[CALCULATE],[nan],...,2.0,True,1.740569e+12,HINT,[],"[{'type': 'HINT', 'main': 'Pas het kwadraat to...",NaN,NaN,NaN,NaN


### Interesting attributes to add
* Number of interactions
* Number of questions
* Number of interactions per question
* Number of events (and count per event type (HINT, EVALUATE, FEEDBACK))
* Count of each questionType, itemType, etc.

In [3]:
df_library.columns

Index(['exerciseID', 'sessionID', 'sessionSuccess', 'creationTimestamp',
       'finished', 'marksEarned', 'marksTotal', 'skillName', 'stepType',
       'errors', 'questionID', 'questionType', 'itemType', 'itemContent',
       'interactionID', 'interactionType', 'interactionSolution',
       'interactionDerivation', 'result.status', 'result.progress',
       'result.events', 'result.tags', 'result.scoring.marksEarned',
       'result.scoring.marksTotal', 'result.scoring.penalties.marksPenalty',
       'result.scoring.penalties.hintsRequested',
       'result.scoring.penalties.mathErrors', 'result.scoring.finished',
       'result.scoring.maxAttempts', 'result.scoring.usedAttempts',
       'result.scoring.failed', 'eventID', 'eventType', 'eventTags',
       'eventAnnotations', 'exerciseStatusAfterEvent', 'progressAfterEvent',
       'inputStatus', 'skillsTodo'],
      dtype='object')

In [4]:
# Lists of attribute types
ids = ['exerciseID', 'sessionID', 'creationTimestamp', 'questionID', 'interactionID', '']
numerical = ['marksEarned', 'marksTotal', 'result.progress', 'result.scoring.marksEarned', 'result.scoring.marksTotal', 'result.scoring.penalties.marksPenalty', 'result.scoring.penalties.hintsRequested', 
             'result.scoring.penalties.mathErrors', 'result.scoring.maxAttempts', 'result.scoring.usedAttempts', 'progressAfterEvent', 'inputStatus']
boolean = ['sessionSuccess', 'finished', 'result.scoring.failed']
categorial = ['questionType', 'itemType', 'interactionType', 'result.status', 'eventType', 'exerciseStatusAfterEvent'] #count how many of each category maybe?
collection = ['skillName', 'stepType', 'skillsTodo'] #list -> One hot encoding for skills?
sentence = ['itemContent', 'interactionSolution'] #string
dictionary = ['errors', 'interactionDerivation', 'result.events', 'result.tags', 'eventTags', 'eventAnnotations']


In [5]:
A = s.Subgroup("Failing", "{a1 = [8, 9]}", [1,3,4,6], 68.5, [41.1, 32.1, 33.0, 42.0])

beam = pq.PriorityQueue([A], 3)
print(beam._average_quality)

beam.summarize_queue()
print(beam._average_quality)

None
This queue has subgroups of average size 4.0 in a queue of length 1 and an average quality of 68.5.
68.5


In [6]:
rf.numerical_refinement(df_library, "marksTotal", 15)

[{'marksTotal': (1.0, 2.0)},
 {'marksTotal': (2.0, 3.0)},
 {'marksTotal': (3.0, 4.0)},
 {'marksTotal': (4.0, 5.0)},
 {'marksTotal': (5.0, 6.0)},
 {'marksTotal': (6.0, 7.0)},
 {'marksTotal': (7.0, 8.0)}]

In [7]:
rf.boolean_refinement(df_library, "finished")

[{'finished': np.False_}, {'finished': np.True_}]

In [8]:
cats = rf.categorical_refinement(df_library, "eventType")
att_list = []
for cas in cats:
    att_list.append(cas['eventType'])

att_list

[(1.0, 'EVALUATE'),
 (0.0, 'EVALUATE'),
 (1.0, 'HINT'),
 (0.0, 'HINT'),
 (1.0, 'GIVEUP'),
 (0.0, 'GIVEUP')]

In [9]:
rf.collection_refinement(df_library, "skillsTodo", 3)

[{'skillsTodo': ('FractionArithmeticMultiplicationProper',)},
 {'skillsTodo': ('FractionArithmeticSimplifyProper',)},
 {'skillsTodo': ('EquationsBalanceMethodDivideCoefficient',)},
 {'skillsTodo': ('EquationsOperationsMultiply',)},
 {'skillsTodo': ('EquationsBalanceMethodArrangeTerms',)},
 {'skillsTodo': ('EquationsOperationsAdd',)},
 {'skillsTodo': ('AlgebraAddingLikeTerms',)},
 {'skillsTodo': ('ExpandSingleBrackets',)},
 {'skillsTodo': ('FractionArithmeticSimilarImproperToMixed',)},
 {'skillsTodo': ('FractionArithmeticSimilarMixed',)},
 {'skillsTodo': ('FractionArithmeticSimilarMixedToImproper',)},
 {'skillsTodo': ('FractionArithmeticDivisionProper',)},
 {'skillsTodo': ('FractionArithmeticMultiplicationMixed',)},
 {'skillsTodo': ('FractionArithmeticAddProperUnlikeDenominator',)},
 {'skillsTodo': ('FractionArithmeticAddProperCommonDenominator',)},
 {'skillsTodo': ('FactorSingleBrackets',)},
 {'skillsTodo': ('FractionArithmeticMultiplicationProper',
   'FractionArithmeticSimplifyProper

In [10]:
df_library['finished'].unique()

array([False,  True])

# Skips
* errors: now a dictionary, should be further processed
* itemContent: too specific, and basically indicates same exercises. Would not advice to include.-> for future if we want to do a string refinement where we check if certain elements are included, e.g. x or y idk.
* interactionSolution: same as above, equations. -> for future if we want to do a string refinement where we check if certain elements are included, e.g. x or y idk.
* interactionDerivation: same as above, equations.-> for future if we want to do a string refinement where we check if certain elements are included, e.g. x or y idk.
* result.events: now a dictionary, should be further processed
* result.tags: now a dictionary, should be further processed
* result.scoring.maxAttempts: always 2
* eventTags: now a dictionary, should be further processed
* eventAnnotations: now a dictionary, should be further processed

In [11]:
descriptive_space, target_space, attributes, descriptor_types = ds.split_spaces(df_library, 
                                                                                descriptors = [ 'skillName', 'stepType','questionType', 'itemType', 'interactionType',
                                                                                                'result.status', 'result.progress', 'result.scoring.penalties.marksPenalty',
                                                                                                'result.scoring.penalties.hintsRequested',
                                                                                                'result.scoring.penalties.mathErrors', 'result.scoring.usedAttempts',
                                                                                                'eventType', 'exerciseStatusAfterEvent', 'progressAfterEvent',
                                                                                                'inputStatus', 'skillsTodo'], 
                                                                                targets = [ 'finished', 'marksEarned', 'marksTotal', 'result.scoring.marksEarned',
                                                                                            'result.scoring.marksTotal', 'result.scoring.failed', 'result.scoring.finished'], # all to do with performance
                                                                                ids = ['exerciseID', 'sessionID', 'creationTimestamp', 'questionID', 'interactionID', 'eventID'],
                                                                                skips=['errors', 'itemContent', 'interactionSolution', 'interactionDerivation', 'result.events',
                                                                                       'result.tags', 'result.scoring.maxAttempts', 'eventTags', 'eventAnnotations']) # skips are to difficult atm
print(descriptor_types)


{'categorical': ['result.status', 'eventType', 'exerciseStatusAfterEvent', 'inputStatus'], 'boolean': [], 'numerical': ['result.progress', 'result.scoring.penalties.marksPenalty', 'result.scoring.penalties.hintsRequested', 'result.scoring.penalties.mathErrors', 'result.scoring.usedAttempts', 'progressAfterEvent'], 'collection': ['skillName', 'stepType', 'skillsTodo']}


In [14]:
candidates_per_attribute, candidates = ds.get_candidates_per_attribute(descriptive_space=descriptive_space, dtypes=descriptor_types, numerical_bins=3, collection_limit=3)
for att in candidates_per_attribute.keys():
    print(f'{att}: {candidates_per_attribute[att]}')

result.progress: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
result.scoring.penalties.marksPenalty: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
result.scoring.penalties.hintsRequested: [(0.0, 2.3333333333333335), (2.3333333333333335, 4.666666666666667), (4.666666666666667, 7.0)]
result.scoring.penalties.mathErrors: [(0.0, 3.0), (3.0, 6.0), (6.0, 9.0)]
result.scoring.usedAttempts: [(0.0, 3.0), (3.0, 6.0), (6.0, 9.0)]
progressAfterEvent: [(0.0, 0.3333333333333333), (0.3333333333333333, 0.6666666666666666), (0.6666666666666666, 1.0)]
result.status: [(1.0, 'FINISHED'), (0.0, 'FINISHED'), (1.0, 'GIVEUP'), (0.0, 'GIVEUP'), (1.0, 'VIRGIN'), (0.0, 'VIRGIN'), (1.0, 'CORRECT'), (0.0, 'CORRECT'), (1.0, 'ERROR'), (0.0, 'ERROR')]
eventType: [(1.0, 'EVALUATE'), (0.0, 'EVALUATE'), (1.0, 'HINT'), (0.0, 'HINT'), (1.0, 'GIVEUP'), (0.0, 'GIVEUP')]
exerciseStatusAfterEvent: [(1.0, 'FINISHED'), (0.0, '